# Find max-winds in a wave-following box
This code uses the wavetrack from the QTRACK tool and retrieves the maximum winds in a wave-following box of a user-specified size. Using code from Anantha, we interpolate the track to match wrf output and retrieve 3 hrly max winds for each ensemble member

In [1]:
from __future__ import print_function
import pkg_resources
import os
#from os import listdir
#from os.path import isfile, join
import glob
import metpy
import subprocess
import wrf
import numpy as np
from netCDF4 import Dataset

import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
import matplotlib.cbook as cbook
import cartopy.crs as crs
import cartopy.feature as cfeature
from metpy.units import units

from cartopy.feature import NaturalEarthFeature

from wrf import (to_np, interplevel, geo_bounds, getvar, smooth2d, get_cartopy, cartopy_xlim,
                 cartopy_ylim, latlon_coords, destagger)
from matplotlib.colors import Normalize
import matplotlib.patches as mpatches


from metpy.plots import ctables

import metpy.calc as mpcalc
import xarray as xr
import pandas as pd

/glade/derecho/scratch/athornton/tmp/ipykernel_34334/2675992302.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
## The following are lists to select each ensemble set and respective initialization time
variation = ['fluxon', 'rst_on24', 'rst_on36', 'rst_on48', '', 'fluxoff', '', '', '']
init_times = ['0300','0303','0306',
             '0309','0312','0315',
             '0318','0321','0400']

In [549]:
#########################################
### MAKE SELECTIONS HERE
#set = 'Fluxes on'          # try: '24 hours without fluxes'
set_name = variation[0]     # try: variation[1]
wave = 'main'               # 'second' == Paulette or 'main' == Rene
ens = init_times[0]

## 
if wave == 'main':
    name = 'Rene'
else:
    name = 'Paulette'

In [550]:
plotsdir = '/glade/u/home/athornton/qtrack/max_winds/'

In [551]:
save_name = set_name +"_"+ ens
save_name

'fluxoff_0400'

## First get the track data
We will first open the track corresponding to our run selections, then interpolate it from 6hrly to 3hrly to match the wrf output

In [552]:
df = xr.open_dataset('/glade/u/home/athornton/qtrack/'+wave+'_wave/'+wave+'_wave_track_'+save_name+'.nc')

In [553]:
# From Anantha...
# Simple linear interpolation between each latitude point
def get_lats():
    new_lats = []
    wave_track_lats = df.lat[-len(times_list):]
    for i in range(0,len(wave_track_lats)):
        if wave_track_lats[i] == wave_track_lats[-1]:
            new_lats.append(wave_track_lats[i].values)
        else:
            next_value = 0.5*(wave_track_lats[i] + wave_track_lats[i+1])
            new_lats.append(wave_track_lats[i].values)
            new_lats.append(next_value.values)
    return new_lats

In [554]:
# From Anantha...
# Simple linear interpolation between each longitude point
def get_lons():
    new_lons = []
    wave_track_lons = df.lon[-len(times_list):] 
    for i in range(0,len(wave_track_lons)):
        if wave_track_lons[i] == wave_track_lons[-1]:
            new_lons.append(wave_track_lons[i].values)
        else:
            next_value = 0.5*(wave_track_lons[i] + wave_track_lons[i+1])
            new_lons.append(wave_track_lons[i].values)
            new_lons.append(next_value.values)
    return new_lons

In [555]:
init_time ='2020-09-'+ens[:2]+' '+ens[2:]+':00'

end_time  ='2020-09-09 12:00'
times_list = pd.date_range(start=init_time, end=end_time, freq='6h')
wave_track_lats = get_lats()
wave_track_lons = get_lons()

## Now get the corresponding WRF run
We will select only the data from the wrf run that corresponds to this wave track, in a 1x1 degree box following the wave. 

In [556]:
# Set directory where wrfout files reside, and list the files for processing.  Set up for a directory with only wrfout files.
os.chdir("/glade/campaign/univ/uncs0067/flux_experiments/cent_atl_case/init"+ens+"z/"+set_name+"/")
plotsdir = "/glade/u/home/athornton/wrf_visualization/plots/restart/init"+ens+"z/"+set_name+"/"
datafiles = sorted(glob.glob("./wrfout*"))
numfiles=len(datafiles)
print(numfiles)
print(datafiles[0])
wrf_out_data = xr.open_dataset(datafiles[0])  
p =wrf_out_data['P']

45
./wrfout_d01_2020-09-04_00:00:00


In [557]:
# Set up loop to run same plots for multiple different times (assume here that we have 1 time per wrfout file)
winds_list = []
for i in range(0,len(datafiles)):
    # for j in range(0,1):
    ncfile = Dataset(datafiles[i])
    Time=wrf.extract_times(ncfile, timeidx=0, method='cat', squeeze=True, cache=None, meta=False, do_xtime=False)
    timestr=(str(Time))
    # Set up one time string for plot titles, another for file names
    titletime=(timestr[0:10]+' '+timestr[11:16])
    filetime=(timestr[0:10]+'_'+timestr[11:16])
    print('WRF valid time: ',filetime)
    
    wind_sfc = getvar(ncfile, "wspd_wdir10")[0]

    # Find the lat,lon of the center of the square we are interested in.
    # For the odd ensemble members, track data ends at 9z. 
    # because our box is big enough to capture the wave/tc, we will keep the lat,lon
    # from the 9z and apply it to 12z, to have wind speeds that cover the length 
    # of the simulation. 
    if i == len(wave_track_lats):
        print('odd ens member, time mismatch. saving track point from previous timestep')
        lat_lon = [wave_track_lats[i-1],wave_track_lons[i-1]]    
    else:
        # Length of wave_track should match number of datafiles for even ensemble members
        lat_lon = [wave_track_lats[i],wave_track_lons[i]]                 
    
    x_ycenter = wrf.ll_to_xy(ncfile, lat_lon[0], lat_lon[1])

    # specify how big you want the wave-following box to be
    delta = 50                                      # in grid points 

    winds = wind_sfc[int(x_ycenter[1]-delta):int(x_ycenter[1]+delta),int(x_ycenter[0]-delta):int(x_ycenter[0]+delta)]

    # wave tracks have nans at the beginning so return nan if it's the beginning of the track
    if not winds.any():
        max_wind = np.nan
    else:
        max_wind = np.max(winds.values)

    winds_list.append(max_wind)
    

WRF valid time:  2020-09-04_00:00
WRF valid time:  2020-09-04_03:00
WRF valid time:  2020-09-04_06:00
WRF valid time:  2020-09-04_09:00
WRF valid time:  2020-09-04_12:00
WRF valid time:  2020-09-04_15:00
WRF valid time:  2020-09-04_18:00
WRF valid time:  2020-09-04_21:00
WRF valid time:  2020-09-05_00:00
WRF valid time:  2020-09-05_03:00
WRF valid time:  2020-09-05_06:00
WRF valid time:  2020-09-05_09:00
WRF valid time:  2020-09-05_12:00
WRF valid time:  2020-09-05_15:00
WRF valid time:  2020-09-05_18:00
WRF valid time:  2020-09-05_21:00
WRF valid time:  2020-09-06_00:00
WRF valid time:  2020-09-06_03:00
WRF valid time:  2020-09-06_06:00
WRF valid time:  2020-09-06_09:00
WRF valid time:  2020-09-06_12:00
WRF valid time:  2020-09-06_15:00
WRF valid time:  2020-09-06_18:00
WRF valid time:  2020-09-06_21:00
WRF valid time:  2020-09-07_00:00
WRF valid time:  2020-09-07_03:00
WRF valid time:  2020-09-07_06:00
WRF valid time:  2020-09-07_09:00
WRF valid time:  2020-09-07_12:00
WRF valid time

In [558]:
time = pd.date_range(start=init_time, end=end_time, freq='3h')

In [559]:
# Create one dataset with u, v winds at each lat, lon, time coordinate for this ensemble member
ds = xr.Dataset( 
    coords=dict(
        time=("time", time),
        max_wind=("max_wind", winds_list),))

In [560]:
# apply regridding and save file
path_out = "/glade/u/home/athornton/qtrack/"+wave+"_wave/"
file_out = path_out +wave+'_wave_max_winds_'+save_name+'.nc'
ds.to_netcdf(path=file_out, format='NETCDF4', mode='w')

In [548]:
# # Set up loop to run same plots for multiple different times (assume here that we have 1 time per wrfout file)
# winds_list = []
# for i in range(0,len(datafiles)):
#     # for j in range(0,1):
#     ncfile = Dataset(datafiles[i])
#     Time=wrf.extract_times(ncfile, timeidx=0, method='cat', squeeze=True, cache=None, meta=False, do_xtime=False)
#     timestr=(str(Time))
#     # Set up one time string for plot titles, another for file names
#     titletime=(timestr[0:10]+' '+timestr[11:16])
#     filetime=(timestr[0:10]+'_'+timestr[11:16])
#     print('WRF valid time: ',filetime)

#     wrf_out_data = xr.open_dataset(datafiles[i])  

#     # Get all the variables we need
#     ua = wrf_out_data["U"]
#     va = wrf_out_data["V"]

#     lats, lons = latlon_coords(wrf_out_data['P'])
#     lats_np = to_np(lats)
#     lons_np = to_np(lons)
#     lats_np = lats_np[0,:,0]
#     lons_np = lons_np[0,0,:]
    
#     # Unstagger the winds to match the pressure grid
#     ua = destagger(ua, stagger_dim=3)             # Unstagger U in the x-direction
#     va = destagger(va, stagger_dim=2)             # Unstagger V in the x-direction
    
#     u_winds = xr.DataArray(ua)[0]
#     v_winds = xr.DataArray(va)[0]
    
#     u_700 = u_winds.sel(dim_1=17)*units('m/s')     # select 700 hPa winds
#     v_700 = v_winds.sel(dim_1=17)*units('m/s')     # select 700 hPa winds
#     u_sfc = u_700 * 0.9                            # calculate surface winds 
#     v_sfc = v_700 * 0.9                            # as in Hill and Lackmann, 2009 
#     wind_sfc = mpcalc.wind_speed(u_sfc, v_sfc)

#     # Find the lat,lon of the center of the square we are interested in.
#     # For the odd ensemble members, track data ends at 9z. 
#     # because our box is big enough to capture the wave/tc, we will keep the lat,lon
#     # from the 9z and apply it to 12z, to have wind speeds that cover the length 
#     # of the simulation. 
#     if i == len(wave_track_lats):
#         print('odd ens member, time mismatch. saving track point from previous timestep')
#         lat_lon = [wave_track_lats[i-1],wave_track_lons[i-1]]    
#     else:
#         # Length of wave_track should match number of datafiles for even ensemble members
#         lat_lon = [wave_track_lats[i],wave_track_lons[i]]                 
    
#     x_ycenter = wrf.ll_to_xy(ncfile, lat_lon[0], lat_lon[1])

#     # specify how big you want the wave-following box to be
#     delta = 50                                      # in grid points 

#     winds = wind_sfc[int(x_ycenter[1]-delta):int(x_ycenter[1]+delta),int(x_ycenter[0]-delta):int(x_ycenter[0]+delta)]

#     # wave tracks have nans at the beginning so return nan if it's the beginning of the track
#     if not winds.any():
#         max_wind = np.nan
#     else:
#         max_wind = np.max(winds.values)

#     winds_list.append(max_wind)
    